In [ ]:
from utils import preprocess
from utils import models
from utils import visualize

df_customers, df_products, df_transactions = preprocess.load_complete_dataset_filtered_number_customers(200)
df_transactions = preprocess.filter_customers_by_activity(df_transactions)

models.xgboost_preprocess(df_customers, df_products, df_transactions, n_negativos_por_positivo=4, random_state=42)


X, y, sample_weight, dataset, article_df, user_df = models.xgboost_preprocess(
    df_customers, df_products, df_transactions,
    n_negativos_por_positivo=4, random_state=42,
    # Ejemplo: bajar el peso de la ropa interior en el entrenamiento
    # category_weights={"product_group_name": {"Underwear": 0.3, "Underwear/nightwear": 0.3}},
)

from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split

X_train, X_val, y_train, y_val, w_train, w_val = train_test_split(
    X, y, sample_weight, test_size=0.2, random_state=42, stratify=y
)

model = XGBClassifier(n_estimators=300, max_depth=6, learning_rate=0.05,
                      use_label_encoder=False, eval_metric='logloss', random_state=42)
model.fit(X_train, y_train, sample_weight=w_train, eval_set=[(X_val, y_val)], verbose=50)
